# Initialize — TPC-H Sample (single catalog)

Shared variables + helpers for the staging notebooks. Invoke via `%run ./initialize`.
Every medallion layer lives in ONE catalog, separated by schema
(`{schema_namespace}_<layer>[_<source>]{logical_env}`), derived from `catalog`,
`schema_namespace` and `logical_env`.

In [ ]:
from datetime import datetime, timedelta
import pyspark.sql.functions as F

dbutils.widgets.text("catalog", "main")
dbutils.widgets.text("schema_namespace", "tpch_sample")
dbutils.widgets.text("logical_env", "")

catalog = dbutils.widgets.get("catalog")
schema_namespace = dbutils.widgets.get("schema_namespace")
logical_env = dbutils.widgets.get("logical_env")

# Single-catalog model: one catalog, one schema per medallion layer (and per bronze source)
staging_schema = f"{schema_namespace}_staging{logical_env}"
silver_schema = f"{schema_namespace}_silver{logical_env}"
gold_schema = f"{schema_namespace}_gold{logical_env}"
staging_volume = "stg_volume"

# Per-source-system bronze schemas
bronze_source_systems = [
    "reference_data", "crm", "procurement", "vendor_mgmt",
    "order_mgmt", "order_fulfillment", "product_catalog", "inventory",
]


def bronze_schema(system):
    return f"{schema_namespace}_bronze_{system}{logical_env}"


sample_source_schema = "samples.tpch"

volume_root = f"/Volumes/{catalog}/{staging_schema}/{staging_volume}"
sources_root = f"{volume_root}/sources"

# Staging uses Parquet (typed, self-describing) so bronze can infer + evolve the schema
# via Auto Loader. No source schema is declared in the bronze flows.

# Landing-zone layout: sources/{system}/{entity}/{zone}/{entity}_<yyyymmddHHMMSS>.parquet
# zone "initial"     -> day-1 baseline (setup job), persists across full refreshes
# zone "incremental" -> day-2/3 batches (run_2/run_3), safe to wipe via reset_incremental()
INITIAL_ZONE = "initial"
INCREMENTAL_ZONE = "incremental"


def landing_dir(system, entity, zone):
    return f"{sources_root}/{system}/{entity}/{zone}"


def write_landing(df, system, entity, zone, load_ts, single_file=None):
    """Write a timestamped Parquet batch into the landing zone.

    Layout:
    - incremental (default single_file=True): one file ``{entity}_{ts}.parquet`` (small batches)
    - initial (default single_file=False): folder ``{entity}_{ts}.parquet/part-*.parquet`` (no coalesce)

    Bronze Auto Loader reads ``sources/{system}/{entity}/`` recursively (initial/ + incremental/).
    """
    if single_file is None:
        single_file = zone != INITIAL_ZONE

    suffix = load_ts.strftime("%Y%m%d%H%M%S")
    target_dir = landing_dir(system, entity, zone)

    if single_file:
        fname = f"{entity}_{suffix}.parquet"
        tmp_dir = f"{target_dir}/_tmp_{fname}"
        df.coalesce(1).write.format("parquet").mode("overwrite").save(tmp_dir)
        part = [f.path for f in dbutils.fs.ls(tmp_dir) if f.name.endswith(".parquet")][0]
        dbutils.fs.mv(part, f"{target_dir}/{fname}")
        dbutils.fs.rm(tmp_dir, True)
        print(f"  wrote {system}/{entity}/{zone}/{fname}")
    else:
        batch_dir = f"{target_dir}/{entity}_{suffix}.parquet"
        df.write.format("parquet").mode("overwrite").save(batch_dir)
        n_files = len([f for f in dbutils.fs.ls(batch_dir) if f.name.endswith(".parquet")])
        print(f"  wrote {system}/{entity}/{zone}/{entity}_{suffix}.parquet/ ({n_files} parquet files)")


def reset_incremental():
    """Remove every incremental/ landing folder so a full refresh reprocesses the
    day-1 initial/ data only (no need to re-stage the initial sample)."""
    reset = 0
    for system in bronze_source_systems:
        try:
            entities = dbutils.fs.ls(f"{sources_root}/{system}")
        except Exception:
            continue
        for e in entities:
            inc = f"{e.path.rstrip('/')}/{INCREMENTAL_ZONE}"
            try:
                dbutils.fs.rm(inc, True)
                reset += 1
            except Exception:
                pass
    print(f"Reset incremental landing zone ({reset} entity folders cleared).")


def with_metadata(df, system, batch_id, load_ts):
    return (df
            .withColumn("load_timestamp", F.lit(load_ts).cast("timestamp"))
            .withColumn("source_system", F.lit(system))
            .withColumn("batch_id", F.lit(batch_id)))


def with_effective(df, effective_date):
    """Stamp a business effective date (the SCD2 sequence_by for the customer/supplier
    dimensions). This is on the business timeline (1992-1998) so gold facts can resolve the
    dimension version that was effective as of the order_date (point-in-time). Distinct from
    load_timestamp, which is processing time."""
    return df.withColumn("effective_date", F.lit(effective_date).cast("timestamp"))


def with_op(df, op="U"):
    """Stamp a CDC operation flag (cdc_operation) carried by every SCD2 dimension feed. 'U' is a
    normal upsert (insert/update); 'D' marks a delete. Silver applies it via
    cdcSettings.apply_as_deletes ("cdc_operation = 'D'") so a delete closes the open SCD2 version
    (tombstone) instead of inserting. The column is in except_column_list, so it drives the
    merge but is never stored in the target."""
    return df.withColumn("cdc_operation", F.lit(op))


# --- Always-on staging economics + SLA shaping (applied to EVERY batch) ------------------------
# These run for the initial load and every incremental run so magnitude and fulfillment behaviour
# stay consistent across batches. They are volume-preserving (row counts unchanged) and independent
# of the optional growth/seasonality/skew reshaping in tpch_data_realism.

# Unit-price reduction: raw TPC-H extended prices aggregate into the trillions, which reads as
# unrealistic on the dashboards/Genie. Scale unit price (l_extendedprice / o_totalprice) by a fixed
# factor so total net sales land near ~$800M. Quantity is untouched (price-per-unit drops).
# Calibrated against the rank-based supplier value weighting in tpch_data_realism (batch-1).
PRICE_SCALE = 4.53e-4

# Ship-mode fulfillment SLA: receipt_date = ship_date + transit(mode). Faster modes arrive before
# the commit date more often, so on-time rate (receipt_date <= commit_date) varies by mode
# (AIR best -> MAIL worst) instead of being identical across modes. (min_days, max_days) per mode.
SHIP_MODE_TRANSIT = {
    "AIR": (1, 3), "REG AIR": (2, 5), "TRUCK": (3, 7), "RAIL": (5, 10),
    "FOB": (7, 13), "SHIP": (10, 20), "MAIL": (12, 25),
}


def scale_line_prices(df):
    """Reduce line-item unit price (volume-preserving; quantity unchanged)."""
    return df.withColumn(
        "l_extendedprice",
        (F.col("l_extendedprice") * F.lit(PRICE_SCALE)).cast("decimal(18,2)"),
    )


def scale_order_totals(df):
    """Reduce order-header total price to match the scaled line economics."""
    return df.withColumn(
        "o_totalprice",
        (F.col("o_totalprice") * F.lit(PRICE_SCALE)).cast("decimal(18,2)"),
    )


def apply_ship_mode_sla(df):
    """Derive receipt_date from ship_date + a mode-specific transit time so on-time delivery
    differs by ship mode. Deterministic by (orderkey, linenumber)."""
    base = F.lit(14)
    span = F.lit(1)
    for mode, (lo, hi) in SHIP_MODE_TRANSIT.items():
        base = F.when(F.col("l_shipmode") == mode, F.lit(lo)).otherwise(base)
        span = F.when(F.col("l_shipmode") == mode, F.lit(hi - lo + 1)).otherwise(span)
    transit = base + F.pmod(F.hash("l_orderkey", "l_linenumber"), span)
    return df.withColumn("l_receiptdate", F.date_add(F.col("l_shipdate"), transit))


def adjust_lineitem_staging(df):
    """Always-on line-item economics + SLA shaping applied to every batch."""
    return apply_ship_mode_sla(scale_line_prices(df))


print("Initialized:")
print(f"  catalog        = {catalog}")
print(f"  staging_schema = {staging_schema}")
print(f"  silver_schema  = {silver_schema}")
print(f"  gold_schema    = {gold_schema}")
print(f"  sources_root   = {sources_root}")